In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("US_Accidents_Analysis") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 13:19:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Load raw CSV
df_raw = spark.read.csv("data/US_Accidents_March23.csv", header=True, inferSchema=False)

# Select columns for Risk Analysis and hotspots
target_cols = [
    'Severity', 'Start_Time', 'Start_Lat', 'Start_Lng', 'Street', 'Zipcode', 'City', 'State',
    'Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Precipitation(in)', 'Weather_Condition',
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 
    'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Sunrise_Sunset'
]

df = df_raw.select(target_cols)

In [3]:
# Preprocessing steps

# 1. cast numerics
df = df.withColumn("Severity", F.col("Severity").cast("integer")) \
       .withColumn("Start_Time", F.to_timestamp("Start_Time")) \
       .withColumn("Temperature(F)", F.col("Temperature(F)").cast("float")) \
       .withColumn("Visibility(mi)", F.col("Visibility(mi)").cast("float"))

# 2. Replace Nulls (Median for weather, "Unknown" for strings)
df = df.fillna({'Temperature(F)': 60.0, 'Visibility(mi)': 10.0, 'Precipitation(in)': 0.0})
df = df.fillna("Unknown", subset=["Weather_Condition", "Sunrise_Sunset"])

# 3. Time Features
df = df.withColumn("Hour", F.hour("Start_Time")) \
       .withColumn("DayOfWeek", F.dayofweek("Start_Time"))

# 4. Binary Road Features & Geo Cleaning
road_features = ['Amenity', 'Bump', 'Crossing', 'Junction', 'Railway', 'Stop', 'Traffic_Signal']
for col in road_features:
    df = df.withColumn(col, F.col(col).cast("integer"))

df = df.withColumn("Zipcode_Short", F.substring(F.col("Zipcode"), 1, 5))

# 5. Group Weather (High Performance Native Spark)
df = df.withColumn("Weather_Simple", 
    F.when(F.col("Weather_Condition").contains("Rain"), "Rain")
     .when(F.col("Weather_Condition").contains("Snow"), "Snow")
     .when(F.col("Weather_Condition").contains("Cloud") | F.col("Weather_Condition").contains("Overcast"), "Cloudy")
     .when(F.col("Weather_Condition").contains("Clear") | F.col("Weather_Condition").contains("Fair"), "Clear")
     .otherwise("Other"))

26/04/15 13:19:33 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
spark.catalog.clearCache()

df.repartition(10).write.mode("overwrite").parquet("data/US_Accidents_Cleaned.parquet")

# Now load the clean version
df = spark.read.parquet("data/US_Accidents_Cleaned.parquet").cache()
print(f"Dataset Ready: {df.count()} rows")

26/04/15 13:19:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
ERROR:root:Exception while sending command.                       (0 + 16) / 23]
Traceback (most recent call last):
  File "/home/hpeteri/Documents/Big Data Processing and Applications/521283S/bdpa-26/venv/lib64/python3.12/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/hpeteri/Documents/Big Data Processing and Applications/521283S/bdpa-26/venv/lib64/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hpeteri/Documen

Py4JError: An error occurred while calling o140.parquet

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/home/hpeteri/Documents/Big Data Processing and Applications/521283S/bdpa-26/venv/lib64/python3.12/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/hpeteri/Documents/Big Data Processing and Applications/521283S/bdpa-26/venv/lib64/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hpeteri/Documents/Big Data Processing and Applications/521283S/bdpa-26/venv/lib64/python3.12/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or re